# Notebook 2 — Feature Engineering
**Projet IndabaX 2026 — Cameroun**

Prérequis : exécuter `01_EDA.ipynb` → génère `../outputs/df_clean.pkl`


In [3]:
import pandas as pd
import numpy as np
import pickle, os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder

HORIZONS     = [1, 7]
TEST_YEAR    = 2025
RANDOM_STATE = 42
print('Imports OK.')


Imports OK.


## Chargement du contexte EDA

In [4]:
df_clean     = pickle.load(open('../outputs/df_clean.pkl',    'rb'))
NUMERIC_COLS = pickle.load(open('../outputs/numeric_cols.pkl','rb'))
villes_ref   = [v for v in ['Maroua','Douala','Ngaoundere'] if v in df_clean['city'].values]
print(f'df_clean chargé : {df_clean.shape}')
print(f'Villes de référence : {villes_ref}')


df_clean chargé : (87240, 33)
Villes de référence : ['Maroua', 'Douala', 'Ngaoundere']


## 1. Feature Engineering
- **Encodage cyclique** : `sin/cos` du jour de l'année et du mois (continuité Jan–Déc)
- **Lags VRI** : 1, 2, 3, 7, 10, 14, 21, 30 jours par ville
- **Fenêtres glissantes** : mean/std/max sur 7, 14, 30 jours
- **Anomalie** : VRI − climatologie mensuelle par ville
- **Note** : `year` est exclu de FEATURE_COLS (data leakage temporel — il sert de critère de split)


In [5]:
def add_features(df):
    out = df.copy().reset_index(drop=True)

    out['day_of_year']  = out['time'].dt.dayofyear
    out['month']        = out['time'].dt.month
    out['year']         = out['time'].dt.year
    out['week_of_year'] = out['time'].dt.isocalendar().week.astype(int)

    # Encodage cyclique
    out['sin_doy']   = np.sin(2 * np.pi * out['day_of_year'] / 365.25)
    out['cos_doy']   = np.cos(2 * np.pi * out['day_of_year'] / 365.25)
    out['sin_month'] = np.sin(2 * np.pi * out['month'] / 12)
    out['cos_month'] = np.cos(2 * np.pi * out['month'] / 12)

    # Lags VRI par ville (transform évite les fuites inter-villes)
    for lag in [1, 2, 3, 7, 10, 14, 21, 30]:
        out[f'VRI_lag_{lag}'] = out.groupby('city')['VRI'].transform(
            lambda x, l=lag: x.shift(l))

    # Rolling VRI par ville
    for w in [7, 14, 30]:
        grp = out.groupby('city')['VRI']
        out[f'VRI_roll{w}_mean'] = grp.transform(lambda x, ww=w: x.rolling(ww, min_periods=1).mean())
        out[f'VRI_roll{w}_std']  = grp.transform(lambda x, ww=w: x.rolling(ww, min_periods=1).std().fillna(0))
        out[f'VRI_roll{w}_max']  = grp.transform(lambda x, ww=w: x.rolling(ww, min_periods=1).max())

    # Anomalie par rapport à la climatologie mensuelle par ville
    clim = (out.groupby(['city','month'])['VRI'].mean().reset_index()
               .rename(columns={'VRI':'VRI_clim_month'}))
    out = out.merge(clim, on=['city','month'], how='left').reset_index(drop=True)
    out['VRI_anomaly'] = out['VRI'] - out['VRI_clim_month']

    # Précipitations cumulées 15 jours
    out['precip_cum15'] = out.groupby('city')['precipitation_sum'].transform(
        lambda x: x.rolling(15, min_periods=1).sum())

    # Température glissante 7 jours
    out['temp_roll7'] = out.groupby('city')['temp_mean'].transform(
        lambda x: x.rolling(7, min_periods=1).mean())

    # Encodage ville
    le = LabelEncoder()
    out['city_enc'] = le.fit_transform(out['city'])

    city_stats = (out.groupby('city')['VRI']
                     .agg(city_vri_mean='mean', city_vri_std='std').reset_index())
    out = out.merge(city_stats, on='city', how='left').reset_index(drop=True)

    # Cibles futures
    for h in HORIZONS:
        out[f'target_t{h}'] = out.groupby('city')['VRI'].transform(
            lambda x, hh=h: x.shift(-hh))

    return out

df_feat = add_features(df_clean)
new_cols = df_feat.shape[1] - df_clean.shape[1]
print(f'Features ajoutées : {new_cols} nouvelles colonnes')
print(f'Dimensions finales : {df_feat.shape}')


Features ajoutées : 34 nouvelles colonnes
Dimensions finales : (87240, 67)


## 2. Split temporel
Split chronologique strict — toute inversion constitue un data leakage.
- **Train** : 2020–2023
- **Validation** : 2024
- **Test** : 2025

`year` est conservé dans les DataFrames de split pour le filtrage mais **exclu** de FEATURE_COLS.


In [6]:
# 'year' exclu de FEATURE_COLS : évite le data leakage temporel
FEATURE_COLS = [
    'city_enc', 'city_vri_mean', 'city_vri_std', 'latitude', 'longitude',
    'VRI_lag_1', 'VRI_lag_2', 'VRI_lag_3', 'VRI_lag_7',
    'VRI_lag_10', 'VRI_lag_14', 'VRI_lag_21', 'VRI_lag_30',
    'VRI_roll7_mean', 'VRI_roll7_std', 'VRI_roll7_max',
    'VRI_roll14_mean', 'VRI_roll30_mean', 'VRI_roll30_std',
    'VRI_clim_month', 'VRI_anomaly',
    'T_opt', 'HR_proxy', 'P_opt', 'S_continu',
    'temp_mean', 'temp_roll7',
    'precipitation_sum', 'precip_cum15',
    'precipitation_hours', 'et0_fao_evapotranspiration',
    'sunshine_duration', 'wind_speed_10m_max',
    'sin_doy', 'cos_doy', 'sin_month', 'cos_month', 'month',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_feat.columns]

def make_split(df, horizon=1):
    target = f'target_t{horizon}'
    req    = FEATURE_COLS + [target, 'city', 'time', 'region', 'year']
    req    = [c for c in req if c in df.columns]
    dm    = df[req].dropna().reset_index(drop=True)
    train = dm[dm['year'] <= 2023].reset_index(drop=True)
    val   = dm[dm['year'] == 2024].reset_index(drop=True)
    test  = dm[dm['year'] == TEST_YEAR].reset_index(drop=True)
    return (train[FEATURE_COLS], train[target],
            val[FEATURE_COLS],   val[target],
            test[FEATURE_COLS],  test[target],
            train, val, test)

X_tr1, y_tr1, X_val1, y_val1, X_te1, y_te1, tr1, v1, te1 = make_split(df_feat, 1)
X_tr7, y_tr7, X_val7, y_val7, X_te7, y_te7, tr7, v7, te7 = make_split(df_feat, 7)

print('Split t+1 :')
print(f'  Train      : {len(X_tr1):>6,} obs | {tr1["time"].min().date()} -> {tr1["time"].max().date()}')
print(f'  Validation : {len(X_val1):>6,} obs | {v1["time"].min().date()} -> {v1["time"].max().date()}')
print(f'  Test       : {len(X_te1):>6,} obs | {te1["time"].min().date()} -> {te1["time"].max().date()}')
print(f'  Features   : {len(FEATURE_COLS)}')
assert tr1['time'].max() < v1['time'].min(),  'Leakage train/val !'
assert v1['time'].max()  < te1['time'].min(), 'Leakage val/test !'
print('\nVérification anti-leakage : OK')


Split t+1 :
  Train      : 57,000 obs | 2020-02-06 -> 2023-12-31
  Validation : 14,640 obs | 2024-01-01 -> 2024-12-31
  Test       : 14,120 obs | 2025-01-01 -> 2025-12-19
  Features   : 38

Vérification anti-leakage : OK


## 3. Sauvegarde — `df_feat.pkl`

In [7]:
os.makedirs('../outputs', exist_ok=True)
pickle.dump(df_feat,     open('../outputs/df_feat.pkl',            'wb'), protocol=4)
pickle.dump(FEATURE_COLS,open('../outputs/feature_cols_list.pkl',  'wb'), protocol=4)
print('Fichiers sauvegardés :')
print('  ../outputs/df_feat.pkl')
print('  ../outputs/feature_cols_list.pkl')
print(f'Shape df_feat : {df_feat.shape} | Features : {len(FEATURE_COLS)}')


Fichiers sauvegardés :
  ../outputs/df_feat.pkl
  ../outputs/feature_cols_list.pkl
Shape df_feat : (87240, 67) | Features : 38
